# Marker-alignment timing diagnostic

**The question.** SYNASC 2026 Reviewer 1 wrote that our reading of the two near-chance EMI recordings as catastrophic failure caused by RF energy coupling into the sensing chain *"lacks supporting impedance or RF measurements"*.

They are right. We have no such measurements. This notebook tests a different explanation, one we *can* measure, and finds it.

---

## The mechanism

`analysis/loader.py` aligns stimulus markers to the EEG sample axis by least-squares fitting

$$\text{acq\_time} \approx m \cdot (\text{arrival\_time} - \text{anchor}) + c$$

over every received UDP packet. Ordinary least squares is the right estimator **when the noise is symmetric**.

Bluetooth packet delay is not symmetric. A packet can arrive *late*; it cannot arrive *early*. Delay is one-sided, and under link contention it becomes heavy-tailed.

In acq-versus-arrival space, a delayed packet sits **below** the true line — its arrival time is inflated relative to its acquisition time. So the true clock is the **upper envelope** of the point cloud, not its least-squares mean. Fit the mean of a one-sided distribution and the line tilts toward the delayed mass, dragging every stimulus marker off the EEG with it.

When delay is tight the bias is invisible. When the tail blows out, alignment fails.

**And the sample counter cannot see any of this.** Zero samples were dropped in these recordings — the finding reported in `DEVIATIONS.md` (2026-06-11) stands. Every sample arrived. What degraded was *when*.

---

## Scope

This analysis alters no reported value. Re-aligning only the recordings that produced inconvenient results would be outcome-dependent correction and is indefensible — if the upper-envelope fit were ever adopted it would have to be applied uniformly to all 168 recordings. This is a diagnostic.

In [ ]:
import os, sys, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
_here = Path('.').resolve()
repo_root = next((p for p in [_here, _here.parent, _here.parent.parent]
                  if (p / 'config.yaml').exists()), _here)
os.chdir(repo_root); sys.path.insert(0, str(repo_root))

%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mne; mne.set_log_level('ERROR')

from analysis.timing_diagnostic import (
    fit_clock, _load_timing, recording_residuals, scan_all_recordings,
    compare_fits, lag_sweep, score_recording, FLAG_RESIDUAL_MS, DERIVED_PIPELINE,
)

OUT = Path('data/derived') / DERIVED_PIPELINE
OUT.mkdir(parents=True, exist_ok=True)
print('repo root:', repo_root)

## 1. What the pathology looks like

Residuals of the OLS clock fit, for sub-19's EMI recording against sub-19's own control. Same subject, same headset, same session, ~30 minutes apart.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for col, (sub, cond) in enumerate([('19', 'control'), ('19', 'emi')]):
    tr, acq = _load_timing(sub, cond)
    m, c = fit_clock(tr, acq, 'ols')
    resid = (acq - (m * tr + c)) * 1000

    ax = axes[0, col]
    ax.plot(tr, resid, lw=0.4, color='#356')
    ax.axhline(0, color='#c33', lw=0.8, ls='--')
    ax.set_title(f'sub-{sub} {cond}: residual over time (std {resid.std():.1f} ms)')
    ax.set_xlabel('time into recording (s)'); ax.set_ylabel('residual (ms)')

    ax = axes[1, col]
    ax.hist(resid, bins=120, color='#9bd', edgecolor='none')
    ax.axvline(0, color='#c33', lw=0.8, ls='--')
    ax.set_yscale('log')
    ax.set_title(f'residual distribution (1st pct {np.percentile(resid,1):.0f} ms)')
    ax.set_xlabel('residual (ms)')
plt.tight_layout(); plt.show()

print('Residual tails run negative in both recordings, indicating late-arriving')
print('packets. In the EMI recording the tail extends to several hundred ms, which')
print('is sufficient to bias the least-squares fit away from the true clock.')

## 2. How common is it, and is it EMI-specific?

This is the question that decides how the finding gets written up. If the fault were EMI-specific it would be a story about interference. If it appears in every condition it is a sporadic link fault that happened to land where it did.

In [ ]:
scan = scan_all_recordings()
scan.to_csv(OUT / 'residual_scan.csv', index=False)

summary = (scan.groupby('condition')['residual_std_ms']
           .agg(median='median', mean='mean', max='max', n='count'))
summary['n_flagged'] = scan.groupby('condition')['flagged'].sum()
display(summary.round(2))

print(f'total recordings: {len(scan)}')
for thr in (30, 50, 90):
    print(f'  residual std > {thr:>2} ms: {(scan.residual_std_ms > thr).sum():>3}')

print(f'\nAll recordings above the {FLAG_RESIDUAL_MS:.0f} ms flag:')
display(scan[scan.flagged].sort_values('residual_std_ms', ascending=False)
        [['subject', 'condition', 'residual_std_ms', 'p1_ms', 'min_ms']].round(1))

Median residuals are near-identical across the four conditions, and flagged recordings occur in all of them. The pathology is therefore consistent with a sporadic link or host timing fault rather than a consequence of the EMI manipulation. EMI shows at most a modest elevation confined to the tail of the distribution, which the present data are not powered to attribute to the manipulation itself.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for i, cond in enumerate(['control', 'chewing', 'emi', 'acoustic']):
    v = scan[scan.condition == cond]['residual_std_ms'].values
    ax.scatter(np.full_like(v, i) + np.random.uniform(-.13, .13, len(v)), v,
               s=16, alpha=.75, color='#468')
ax.axhline(FLAG_RESIDUAL_MS, color='#c33', ls='--', lw=1,
           label=f'{FLAG_RESIDUAL_MS:.0f} ms flag')
ax.set_yscale('log'); ax.set_xticks(range(4))
ax.set_xticklabels(['control', 'chewing', 'EMI', 'acoustic'])
ax.set_ylabel('clock-fit residual std (ms, log)')
ax.set_title('Timing pathology is rare and occurs in every condition')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 3. Is a constant offset the explanation?

The simplest version of the hypothesis is that markers are shifted by a fixed amount. We test it by sweeping a constant lag and watching AUC.

`sub-01 emi` is the control case — a healthy recording. Watch what happens to it at plus or minus one SOA (233 ms).

In [ ]:
LAGS = [-466, -233, -117, 0, 117, 233, 466]
sweeps = {f'sub-{s} {c}': lag_sweep(s, c, LAGS) for s, c in
          [('01', 'emi'), ('19', 'emi'), ('33', 'emi')]}

fig, ax = plt.subplots(figsize=(7, 3.8))
for label, df in sweeps.items():
    ax.plot(df.lag_ms, df.auc, marker='o', label=label)
ax.axhline(0.5, color='#888', ls=':', lw=1)
ax.axvline(0, color='#c33', ls='--', lw=.8)
ax.set_xlabel('constant marker shift (ms)'); ax.set_ylabel('AUC')
ax.set_title('Constant-lag sweep')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

for label, df in sweeps.items():
    print(f'{label:14s} ' + '  '.join(f'{a:.3f}' for a in df.auc))

Two observations follow.

First, the healthy recording (sub-01) peaks sharply at zero lag and falls below chance at plus or minus one stimulus-onset asynchrony. Below-chance AUC is diagnostic: additive noise drives AUC toward 0.5 and cannot pass it, whereas a one-flash marker offset can, because the epoch then contains the neighbouring flash's evoked response. The EMI recording of sub-19 sits at that same below-chance value with no shift applied.

Second, no constant lag restores performance for sub-19 or sub-33. The misalignment is therefore time-varying rather than a fixed offset, which is the behaviour predicted by one-sided delay and the reason the correction is applied to the clock fit rather than to the marker positions.

## 4. The recovery test

Refit the clock to the upper envelope, hand the corrected markers to the **registered** `preprocess_recording()` (same filtering, epoching, baseline, boundary rejection, ±150 µV gate), and score with that subject's saved v3 model.

If the diagnosis is correct, the collapsed recordings recover and healthy recordings are unaffected. Healthy recordings are included precisely so that this second condition can be tested: a correction that altered them as well would indicate a change in the estimator rather than the repair of a fault.

In [ ]:
PAIRS = [('19', 'emi'), ('33', 'emi'),            # the two collapses
         ('04', 'acoustic'),                       # 2nd-worst residual in study
         ('01', 'emi'), ('05', 'emi'),             # healthy EMI
         ('19', 'control'), ('19', 'acoustic'),    # same subject, healthy conds
         ('33', 'control'), ('33', 'chewing')]

cmp = compare_fits(PAIRS)
cmp.to_csv(OUT / 'realignment_comparison.csv', index=False)
display(cmp)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
labels = [f"{r.subject}\n{r.condition}" for r in cmp.itertuples()]
x = np.arange(len(cmp))
ax.bar(x - .19, cmp.auc_ols, .38, label='OLS fit (as published)', color='#c96')
ax.bar(x + .19, cmp.auc_envelope, .38, label='upper-envelope fit', color='#468')
ax.axhline(0.5, color='#888', ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7.5)
ax.set_ylabel('AUC'); ax.set_title('Re-alignment recovers the collapses, leaves healthy recordings alone')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

broken = cmp[cmp.auc_ols < 0.75]
healthy = cmp[cmp.auc_ols >= 0.75]
print(f'collapsed recordings  (n={len(broken)}): mean AUC change {broken.auc_delta.mean():+.3f}')
print(f'healthy recordings    (n={len(healthy)}): mean AUC change {healthy.auc_delta.mean():+.3f}')
print('\nRecovery is confined to the collapsed recordings; healthy recordings are')
print('unchanged within rounding, as a correct diagnosis predicts.')

## 5. Consequence for the acoustic limitation

The one near-chance acoustic score has been attributed to a participant with professional audio training who reported the stimulus as aversive. That participant's acoustic recording also carries the second-worst marker alignment in the dataset, giving two candidate explanations for the same observation. The re-alignment test discriminates between them.

In [ ]:
s4 = scan[scan.subject == '04'][['condition', 'residual_std_ms', 'p1_ms']].round(1)
display(s4)
row = cmp[(cmp.subject == '04') & (cmp.condition == 'acoustic')]
display(row)
print('Substantial recovery under re-alignment indicates that the behavioural')
print('account and the timing artifact are confounded for this participant.')

## 6. Summary for the write-up

In [ ]:
out = {
    'n_recordings': int(len(scan)),
    'median_residual_ms_by_condition':
        scan.groupby('condition')['residual_std_ms'].median().round(2).to_dict(),
    'n_flagged_by_condition':
        scan.groupby('condition')['flagged'].sum().astype(int).to_dict(),
    'n_above_30ms': int((scan.residual_std_ms > 30).sum()),
    'n_above_90ms': int((scan.residual_std_ms > 90).sum()),
    'recovery': cmp.to_dict('records'),
    'mean_auc_change_collapsed': round(float(broken.auc_delta.mean()), 3),
    'mean_auc_change_healthy': round(float(healthy.auc_delta.mean()), 3),
    'interpretation': (
        'Sporadic one-sided Bluetooth packet delay biases the least-squares '
        'clock fit and misaligns stimulus markers. Rare (11/168 recordings above '
        '30 ms residual), present in all four conditions, and invisible to the '
        'sample-counter integrity check because no samples are dropped. Refitting '
        'the clock to the upper envelope recovers the two collapsed EMI recordings '
        'and leaves healthy recordings unchanged. Reported as a diagnostic; no '
        'published value is altered.'),
}
(OUT / 'timing_diagnostic_summary.json').write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2)[:1800])
print('\nsaved ->', OUT / 'timing_diagnostic_summary.json')

## 7. What produces the misalignment?

Four candidate causes each predict a specific association with the flagged recordings:

| Candidate | Prediction |
|---|---|
| Host scheduling stall | flagged recordings drop stimulus frames |
| Reduced transmit power | flagged recordings show lower headset battery |
| Progressive session wear | flagged recordings fall late in the session |
| The noise manipulation | flagged recordings concentrate in one condition |

The stimulus loop and the UDP receiver share a machine, so the first prediction is
the sharpest available test: a scheduling stall long enough to delay packets by
hundreds of milliseconds would drop tens of frames at 60 Hz.

In [ ]:
from analysis.timing_diagnostic import build_cause_frame, cause_analysis

cause = build_cause_frame()
cause.to_csv(OUT / 'cause_frame.csv', index=False)
res = cause_analysis(cause)

print(f"recordings: {res['n_recordings']}   flagged: {res['n_flagged']}   "
      f"samples dropped across dataset: {res['samples_dropped_total']:.0f}\n")

for name, key in [('host scheduling (late frames)', 'host_scheduling'),
                  ('transmit power (battery)', 'transmit_power')]:
    d = res[key]
    print(f"{name}")
    print(f"   flagged mean {d['flagged_mean']:.3f} vs normal {d['normal_mean']:.3f}")
    print(f"   rho vs residual = {d['spearman_rho_vs_residual']:+.3f} "
          f"(p = {d['spearman_p']:.3g}); Mann-Whitney p = {d['mannwhitney_p']:.3g}\n")

pw = res['progressive_wear']
print('progressive session wear')
print(f"   flagged by session position: {pw['flagged_by_condition_order']}")
print(f"   all recordings by position:  {pw['all_by_condition_order']}")
print(f"   rho vs residual = {pw['spearman_rho_vs_residual']:+.3f}")

None of the four candidates is supported. Frame drops are flat and marginally
*lower* in flagged recordings, battery is indistinguishable, flagged recordings are
spread across all four session positions, and condition medians were shown to be
equal in section 2.

What does appear is clustering at the level of the session.

In [ ]:
sc = res['session_clustering']
print(f"subjects with at least one flagged recording: {sc['subjects_with_any']}")
print(f"subjects with two or more: {sc['n_subjects_with_two_or_more']} "
      f"(expected {sc['expected_if_independent']} if flagged recordings were independent)\n")

multi = [s for s, n in sc['subjects_with_any'].items() if n >= 2]
for s in multi:
    sub = (cause[cause.subject == s]
           .sort_values('condition_order')
           [['condition_order', 'condition', 'residual_std_ms', 'battery_mean', 'flagged']])
    print(f'sub-{s}')
    print(sub.to_string(index=False), '\n')

Within an affected session the flagged recordings form a **contiguous run** that
begins and ends mid-session: sub-04 escalates across three consecutive recordings
and then returns to baseline, sub-10 is clean, degrades for two recordings and
recovers, and sub-25 degrades for two and then clears.

This is the signature of an episodic degradation of the wireless path, persisting
across one to three recordings before resolving. It is not a property of the
headset, the host, the participant, or the condition.

The clustering itself is suggestive rather than established: three subjects show
two or more affected recordings against roughly one expected under independence,
which at these counts corresponds to *p* of about 0.09. The contiguous-run pattern
is reported descriptively.

The specific trigger cannot be identified from the available instrumentation.
Candidates that the present logging cannot distinguish include an intermittent
2.4 GHz emitter in the room, occlusion or displacement of the receiving dongle,
and host USB power management. None was recorded during data collection.

The practical consequence does not depend on resolving this. The clock-fit
residual is already computed inside `analysis/loader.py`; reporting it per
recording, and gating on it, detects the fault regardless of its origin.

## 8. Full-dataset sweep

Sections 4 and 5 established the mechanism on a small hand-picked set: the three
recordings with the largest residuals, plus six healthy recordings as a control.
That is sufficient to demonstrate the mechanism and insufficient to characterise
it. Three questions remain open on that evidence:

1. Do the other eight flagged recordings recover, or were only the extreme three
   ever affected?
2. Does the correction leave *all* healthy recordings alone, or only the six that
   were sampled?
3. Where should a residual-based alarm threshold actually sit? The 30 ms flag used
   above is arbitrary, and at least one recording (sub-25 control, 66.5 ms)
   scored normally despite exceeding it.

The sweep scores every recording under both fits. Run it first:

```bash
python -m analysis.run_timing_sweep --workers 6
```

Resumable, roughly two to three minutes on six cores. Control recordings are
scored by a model trained on their own epochs and are therefore marked
`confounded`; they are excluded from the recovery claim below.

In [ ]:
from analysis.timing_diagnostic import load_sweep

sweep = load_sweep()
if sweep.empty:
    raise RuntimeError('No sweep results. Run: python -m analysis.run_timing_sweep --workers 6')

print(f'recordings swept: {len(sweep)}')
print(f'  flagged (>30 ms): {int(sweep.flagged.sum())}')
print(f'  control (confounded by design): {int(sweep.confounded.sum())}')
sweep.to_csv(OUT / 'full_sweep.csv', index=False)

usable = sweep[~sweep.confounded]
flagged = usable[usable.flagged]
healthy = usable[~usable.flagged]
print(f'\nnon-control recordings: {len(usable)}  '
      f'({len(flagged)} flagged, {len(healthy)} healthy)')

### 8.1 Does the correction disturb healthy recordings?

The claim in section 4 rested on six recordings. This is the same claim on the
full healthy set.

In [ ]:
print('AUC change on healthy non-control recordings (n = %d)' % len(healthy))
print(healthy.auc_delta.describe(percentiles=[.05, .5, .95]).round(4).to_string())
print(f'\n|AUC change| > 0.02: {int((healthy.auc_delta.abs() > 0.02).sum())} of {len(healthy)}')
print(f'|AUC change| > 0.05: {int((healthy.auc_delta.abs() > 0.05).sum())} of {len(healthy)}')

print('\nBalanced-accuracy change on the same set (percentage points):')
print(healthy.balacc_delta.describe(percentiles=[.05, .5, .95]).round(2).to_string())
print('\nBalanced accuracy is threshold-dependent and moves more than AUC even')
print('where the ranking is unchanged; AUC is the cleaner readout of signal.')

### 8.2 Recovery among flagged recordings

In [ ]:
cols = ['subject','condition','residual_std_ms','auc_ols','auc_envelope',
        'auc_delta','balacc_ols','balacc_envelope']
display(flagged.sort_values('residual_std_ms', ascending=False)[cols].round(3))

recovered = flagged[flagged.auc_delta > 0.10]
print(f'flagged non-control recordings: {len(flagged)}')
print(f'  substantially recovered (AUC gain > 0.10): {len(recovered)}')
print(f'  essentially unchanged:                     {len(flagged) - len(recovered)}')
if len(recovered):
    print(f'  residual range among recovered: '
          f'{recovered.residual_std_ms.min():.1f} - {recovered.residual_std_ms.max():.1f} ms')
if len(flagged) > len(recovered):
    unrec = flagged[flagged.auc_delta <= 0.10]
    print(f'  residual range among unrecovered: '
          f'{unrec.residual_std_ms.min():.1f} - {unrec.residual_std_ms.max():.1f} ms')

### 8.3 Where does the alarm threshold belong?

A useful threshold separates recordings whose alignment error changes the result
from those where it does not. Plotting residual magnitude against the AUC change
under correction locates it empirically rather than by assertion.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
ax.scatter(usable.residual_std_ms, usable.auc_delta, s=26, alpha=.75,
           color='#468', label='non-control')
conf = sweep[sweep.confounded]
ax.scatter(conf.residual_std_ms, conf.auc_delta, s=26, alpha=.5,
           color='#c96', marker='^', label='control (confounded)')
ax.axhline(0, color='#888', lw=.8, ls=':')
ax.axvline(FLAG_RESIDUAL_MS, color='#c33', ls='--', lw=1,
           label=f'{FLAG_RESIDUAL_MS:.0f} ms flag (arbitrary)')
ax.set_xscale('symlog', linthresh=10)
ax.set_xlabel('clock-fit residual std (ms)')
ax.set_ylabel('AUC change under correction')
ax.set_title('Residual magnitude vs consequence')
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

for thr in (20, 30, 50, 70, 90, 120):
    above = usable[usable.residual_std_ms > thr]
    hit = above[above.auc_delta > 0.10]
    print(f'  threshold {thr:>3} ms: flags {len(above):>3} recordings, '
          f'{len(hit)} of them materially affected '
          f'(precision {100*len(hit)/len(above) if len(above) else 0:.0f}%)')

### 8.4 What a uniform correction would do to the reported means

Reported values are produced by the registered pipeline and are not altered by
this analysis. This cell estimates only what would change if the corrected fit
were adopted for every recording, which is the input to that decision rather than
the decision itself.

The estimate is approximate in one respect: control recordings would also need
their classifiers retrained on re-aligned epochs, which this sweep does not do.
The control column is therefore indicative only.

In [ ]:
est = (sweep.groupby('condition')[['balacc_ols','balacc_envelope']]
       .mean().round(2))
est['delta_pp'] = (est.balacc_envelope - est.balacc_ols).round(2)
est['n'] = sweep.groupby('condition').size()
est['confounded'] = est.index == 'control'
display(est)

print('Non-control conditions only:')
for c in ['chewing','emi','acoustic']:
    sub = sweep[sweep.condition == c]
    print(f'  {c:9s} {sub.balacc_ols.mean():5.2f} -> {sub.balacc_envelope.mean():5.2f} '
          f'({sub.balacc_envelope.mean() - sub.balacc_ols.mean():+.2f} pp, n={len(sub)})')

### 8.5 Sweep summary

In [ ]:
summary = {
    'n_swept': int(len(sweep)),
    'n_flagged': int(sweep.flagged.sum()),
    'n_control_confounded': int(sweep.confounded.sum()),
    'healthy_noncontrol': {
        'n': int(len(healthy)),
        'auc_delta_mean': round(float(healthy.auc_delta.mean()), 4),
        'auc_delta_p05': round(float(healthy.auc_delta.quantile(.05)), 4),
        'auc_delta_p95': round(float(healthy.auc_delta.quantile(.95)), 4),
        'n_auc_change_over_0.05': int((healthy.auc_delta.abs() > 0.05).sum()),
        'balacc_delta_mean_pp': round(float(healthy.balacc_delta.mean()), 2),
    },
    'flagged_noncontrol': {
        'n': int(len(flagged)),
        'n_recovered_auc_gt_0.10': int((flagged.auc_delta > 0.10).sum()),
        'records': flagged[cols].round(3).to_dict('records'),
    },
    'condition_means_if_adopted': est.drop(columns='confounded').to_dict('index'),
}
(OUT / 'full_sweep_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2)[:1500])
print('\nsaved ->', OUT / 'full_sweep_summary.json')

## 9. Choice of estimator

The upper-envelope fit used above targets the low-delay boundary directly, which
is the physically correct target when delay is one-sided. It is also the most
aggressive of the available options, and section 8 showed it degrades clean
recordings. The standard alternatives for a regression contaminated by a skewed
tail are Huber regression and RANSAC, neither of which was tried before adopting
the envelope fit. This section corrects that omission.

In [ ]:
from analysis.timing_diagnostic import compare_estimators, keep_sensitivity, residual_shape

PAIRS = [('19','emi'), ('33','emi'), ('04','acoustic'),      # severe misalignment
         ('25','chewing'), ('31','chewing'), ('23','chewing'),# degraded by envelope
         ('01','emi'), ('05','emi')]                          # well aligned

est = compare_estimators(PAIRS)
est.to_csv(OUT / 'estimator_comparison.csv', index=False)
display(est.round(3))

Three results follow.

RANSAC returns the least-squares fit unchanged on every recording: with tens of
thousands of packets the delayed minority is treated as inliers and the slope is
unaffected. It is not useful for this problem.

Huber recovers two of the three severely misaligned recordings about as well as
the envelope fit, while leaving well-aligned recordings essentially untouched and
costing far less on the recordings the envelope fit degrades. It is the better
default.

One recording resists Huber. Huber remains a central-tendency estimator, so where
the delay distribution is shifted rather than merely tailed it moves too little.
Only the envelope fit, which targets the boundary, recovers it.

In [ ]:
ks = keep_sensitivity([('19','emi'), ('33','emi'), ('04','acoustic'),
                       ('25','chewing'), ('31','chewing'), ('01','emi')])
ks.to_csv(OUT / 'keep_sensitivity.csv', index=False)
display(ks.round(3))

keepcols = [c for c in ks.columns if c.startswith('keep_')]
span = (ks[keepcols].max(axis=1) - ks[keepcols].min(axis=1))
for (_, r), s in zip(ks.iterrows(), span):
    print(f"sub-{r['subject']} {r['condition']:9s} resid={r['residual_std_ms']:6.1f}ms  "
          f"AUC span across keep = {s:.3f}")

The recovery of the severely misaligned recordings is insensitive to the `keep`
fraction; the degradation of well-aligned recordings is not, and shrinks
monotonically as the fit becomes less aggressive. The substantive finding does not
rest on the unmotivated default of 0.30, while the side effect does.

In [ ]:
shape = residual_shape()
shape.to_csv(OUT / 'residual_shape.csv', index=False)

sweep2 = sweep.merge(shape, on=['subject','condition'])
nc = sweep2[~sweep2.confounded]
from scipy.stats import spearmanr
r_all, p_all = spearmanr(nc['residual_skew'], nc['auc_delta'])
healthy2 = nc[~nc.flagged]
r_h, p_h = spearmanr(healthy2['residual_skew'], healthy2['auc_delta'])
r_n, p_n = spearmanr(nc['n_epochs_ols'], nc['auc_delta'])

print(f'residual skew vs AUC change, non-control : rho={r_all:+.3f} p={p_all:.3g} (n={len(nc)})')
print(f'residual skew vs AUC change, well-aligned: rho={r_h:+.3f} p={p_h:.3g} (n={len(healthy2)})')
print(f'epoch count   vs AUC change              : rho={r_n:+.3f} p={p_n:.3g}')
print()
for lab, sel in [('recovered (dAUC > +0.10)', nc[nc.auc_delta > 0.10]),
                 ('degraded  (dAUC < -0.02)', nc[nc.auc_delta < -0.02]),
                 ('unchanged', nc[(nc.auc_delta >= -0.02) & (nc.auc_delta <= 0.10)])]:
    print(f'  {lab:26s} n={len(sel):>3}  mean residual skew = {sel["residual_skew"].mean():+.2f}')

A plausible account of the degradation seen in section 8 was that the envelope
fit helps where residuals are strongly one-sided and harms where they are close
to symmetric, so that its effect should track residual skewness. It does not.
Degraded recordings are, if anything, *more* skewed than recovered ones, and
surviving epoch count explains nothing either.

The pattern is therefore recorded as unexplained. No mechanism is claimed for it.